In [ ]:
# !pip install torch transformers datasets unsloth langchain langchain-community langchain-core langchain-google-genai google-generativeai chromadb pypdf sentence-transformers scikit-learn numpy sacrebleu evaluate accelerate bitsandbytes rouge_score

In [ ]:
from unsloth import FastLanguageModel
import torch

import json
import evaluate # Import the evaluate library
from sacrebleu.metrics import BLEU

In [ ]:
# Model configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load the model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/TATA/trained_model",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

Unsloth: Will load /content/drive/MyDrive/TATA/trained_model as a legacy tokenizer.
Unsloth 2026.4.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
prompt_template = """Below is a prompt that describes any question a user has or a problem being faced by the user. Write a response that appropriately helps the user answer his question or give the steps to troubleshoot his problem.

### userPrompt:
{}

### Response:
{}"""

In [ ]:
from tqdm.auto import tqdm
from nltk.translate.bleu_score import corpus_bleu
from rouge_score import rouge_scorer

dataset_path = "/content/drive/MyDrive/TATA/final.json"
with open(dataset_path, "r") as f:
    dataset = json.load(f)

predictions = []
references = []

print(f"Evaluating {len(dataset)} samples...")

# Generate predictions for each sample in the dataset
for i, sample in enumerate(tqdm(dataset)):
    user_input = sample.get("instruction", sample.get("prompt", ""))
    ground_truth = sample.get("output", sample.get("response", ""))

    # Format the prompt
    prompt = prompt_template.format(user_input, "")

    # Tokenize and Generate
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extract Response
    response_marker = "### Response:\n"
    if response_marker in generated_text:
        response_start = generated_text.find(response_marker) + len(response_marker)
        pred = generated_text[response_start:].strip()
    else:
        pred = generated_text.strip()

    predictions.append(pred.lower())
    references.append(ground_truth.strip().lower())

# --- Compute Corpus-Level Metrics ---

# BLEU (expects list of lists for references)
bleu_ref = [[r.split()] for r in references]
bleu_pred = [p.split() for p in predictions]
corpus_bleu_score = corpus_bleu(bleu_ref, bleu_pred)

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge1_scores = []
rougeL_scores = []

for p, r in zip(predictions, references):
    scores = scorer.score(r, p)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
avg_rougeL = sum(rougeL_scores) / len(rougeL_scores)

Evaluating 463 samples...


  0%|          | 0/463 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=20


--- Final Evaluation Results ---
BLEU Score: 0.9942
Average ROUGE-1: 0.9992
Average ROUGE-L: 0.9992


In [ ]:
print("\n--- Final Evaluation Results ---")
print(f"BLEU Score: {corpus_bleu_score:.4f}")
print(f"Average ROUGE-1: {avg_rouge1:.4f}")
print(f"Average ROUGE-L: {avg_rougeL:.4f}")

In [ ]:
# def generate(final_prompt: str):
#     # global final_prompt
#     # Extract the prompt from the incoming JSON payload
#     # print("Hello")
#     inputs = tokenizer(
#         [
#             prompt_template.format(
#                 final_prompt,  # instruction
#                 "",  # output - leave this blank for generation!
#             )
#         ], return_tensors="pt").to("cuda")

#     outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
#     # Decode the output
#     generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

#     # Extract only the newly generated response (removing the prompt part)
#     # This assumes your template ends with "### Response:\n"
#     response_start = generated_text.find("### Response:\n") + len("### Response:\n")
#     pred = generated_text[response_start:].strip()
#     return generated_text

In [ ]:
# pred = generate("In Tata Sumo Gold, how is the rear registration plate illuminated in the vehicle?")
# print(pred)

Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


The rear registration plate is illuminated by two concealed lamps.


In [ ]:
# refs = "The rear registration plate is illuminated by two concealed lamps."

In [ ]:
# from nltk.translate.bleu_score import sentence_bleu
# from rouge_score import rouge_scorer

# reference = refs
# candidate = pred

# # BLEU
# bleu_score = sentence_bleu([reference.split()], candidate.split())

# # ROUGE
# scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
# rouge_scores = scorer.score(reference, candidate)

# print("BLEU:", bleu_score)
# print("ROUGE:", rouge_scores)

BLEU: 1.0
ROUGE: {'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0), 'rouge2': Score(precision=1.0, recall=1.0, fmeasure=1.0), 'rougeL': Score(precision=1.0, recall=1.0, fmeasure=1.0)}
